In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from pathlib import Path

# Weather

In [ ]:
projectRoot = Path.cwd().parents[1]
dataPath = Path(projectRoot, 
                "data/raw/") # Replace with the path of your data
taxiPath = Path(dataPath, "tlc_trip_record/yellow/")
fhvhvPath =  Path(dataPath, "tlc_trip_record/fhvhv/")

In [ ]:
ids = pd.read_csv(projectRoot / "data/NYC_Taxi_Zones_20251203.csv")

In [ ]:
dfs=[]
for i in range(1,264):
    try:
        c = pd.read_parquet(dataPath / f"weather/hourly/{i}/2022-01-01_2025-12-31.parquet")
        c["id"] = i
        dfs.append(c)
    except:
        continue
weather = pd.concat(dfs)
#weather["datetime"] = pd.to_datetime(weather["datetime"], utc=True)

In [ ]:
weather.weather_code = weather.weather_code.astype("int")
weather.datetime =weather.datetime.dt.tz_localize(None).astype('datetime64[us]')
weather

In [ ]:
import geopandas as gpd
from shapely import wkt
try:
    geo = pd.read_csv("../../data/NYC_Taxi_Zones_20251203.csv")  # Columna 'geometry' con MULTIPOLYGON WKT
except: 
    print("Warning, can't find file 'data/NYC_Taxi_Zones_20251203.csv'")
geo = geo.rename(columns ={"Shape Geometry" : "geometry", "Shape Length" : "length", "Shape Area" : "area", "Location ID" : "id", "Zone": "zone", "Borough" : "borough"})
# Convierte WKT → geometría
geo['geometry'] = geo['geometry'].apply(wkt.loads) 
gdf = gpd.GeoDataFrame(geo, crs='EPSG:4326')

gdf_proj = gdf.to_crs('EPSG:2263')
gdf_proj['centroid'] = gdf_proj.geometry.centroid
gdf['lat'] = gdf_proj.centroid.to_crs('EPSG:4326').y
gdf['lon'] = gdf_proj.centroid.to_crs('EPSG:4326').x

In [ ]:
import plotly.express as px
fig = px.choropleth_map(
    gdf, # gdf es un dataframe de GeoPandas
    geojson=gdf.geometry, # Una columna con datos de forma MULTIPOLYGON
    locations=gdf.index,   # Usamos el índice para alinear (0, 1, 2...)
    color="borough",       # Colorear por 'borough' (Bronx, Queens...)
    # O podrías usar color="area" para ver tamaño
    hover_name="zone",     # Al pasar el ratón mostrará "Newark Airport", etc.
    hover_data=["id", "area"], # Datos extra al pasar el ratón, se indica el nombre de las columnas
    #map_style="carto-positron", # Fondo limpio
    map_style="open-street-map",  # Este es el estilo clásico de Folium/OSM
    opacity=0.6,
    center={"lat": 40.7128, "lon": -74.0060}, # Centro de NYC
    zoom=9
)

fig.update_layout(
        margin={"r":0,"t":0,"l":0,"b":0}, 
        width=1200,
        height=800,)
fig.show()

In [ ]:
print(f"Rain = 0 : {weather[weather.rain == 0.0].rain.count() / len(weather)}")
print(f"Rain > 0  y  <= 2 : {weather[(weather.rain > 0.0) & (weather.rain <= 2.0)].rain.count()/ len(weather)}")
print(f"Rain > 2 y <= 5: {weather[(weather.rain > 2 )& (weather.rain <= 5.0)].rain.count()/ len(weather)}")
print(f"Rain > 5 y <= 15 : {weather[(weather.rain > 10.0) & (weather.rain <= 15.0)].rain.count()/ len(weather)}")
print(f"Rain > 15 : {weather[(weather.rain > 15.0)].rain.count()/ len(weather)}")

# Taxis yellows

In [ ]:
dfs  = []
years = range(2022,2026,1)
months = [ f"{m:02d}" for m in range(1,13,1)]
for year in years:
    for month in months:
        filePath = Path(taxiPath, "yellow_tripdata_" + f"{year}-{month}.parquet") 
        if (not filePath.exists() or not filePath.is_file()):
            continue
        try:
            d = pd.read_parquet(filePath, columns = ["tpep_pickup_datetime", "PULocationID"])
        except FileNotFoundError:
            print(f"File {filePath} doesn't exist")
            continue
        dfs.append(d)
taxi = pd.concat(dfs)
taxi['tpep_pickup_datetime'] = taxi['tpep_pickup_datetime'].dt.floor('h')

In [ ]:
# Filtramos los unknown
taxi = taxi[ taxi.PULocationID < 264]
taxi = taxi[(taxi.tpep_pickup_datetime.dt.year >= 2022) & (taxi.tpep_pickup_datetime.dt.year < 2026) ]
taxi = taxi.groupby([taxi.tpep_pickup_datetime, taxi.PULocationID]).size().reset_index(name='num_viajes')

In [ ]:
taxi_weather = taxi.merge(
    weather[['datetime', 'id', 'rain']], 
    left_on=['tpep_pickup_datetime', 'PULocationID'],         
    right_on=['datetime', 'id'],            
    how='left'                                      
)
taxi_weather = taxi_weather.dropna()
taxi_weather = taxi_weather.drop("id",axis= 1)

In [ ]:
taxi_weather

In [ ]:
bins = [-1, 0, 2.5, 15, np.inf]
labels = ['Sin lluvia', 'Lluvia ligera', 'Lluvia moderada', 'Lluvia intensa']
taxi_weather['rain_type'] = pd.cut(
    taxi_weather['rain'],
    bins=bins,
    labels=labels
)
taxi_weather

In [ ]:
df = taxi_weather.groupby("rain_type")['num_viajes'].sum().reset_index(name='freq')
# Crear figura
fig = go.Figure(
    data=[
        go.Pie(
            labels=df['rain_type'],  # MODIFICAR: Columna para etiquetas
            values=df['freq'],     # MODIFICAR: Columna para valores
            hole=0.3,                 # MODIFICAR: Tamaño del agujero (0-1, 0 para pie completo)
            textinfo='label+percent', # MODIFICAR: Información a mostrar ('label', 'percent', 'value', 'label+percent', etc.)
            insidetextorientation='radial',  # MODIFICAR: Orientación del texto ('horizontal', 'radial', 'tangential')
            pull=[0, 0.1, 0, 0, 0],   # MODIFICAR: Separación de sectores (lista con valores 0-1)
            marker=dict(
                colors=['#636EFA', '#00CC96', '#EF553B','#AB63FA'],  # MODIFICAR: Colores personalizados
                line=dict(color='#FFFFFF', width=2)  # MODIFICAR: Borde de los sectores
            ),
            rotation=90,              # MODIFICAR: Ángulo de rotación en grados
            direction='clockwise',    # MODIFICAR: Dirección ('clockwise', 'counterclockwise')
            sort=True                 # MODIFICAR: Ordenar sectores por valor
        )
    ]
)

# MODIFICAR: Personaliza el diseño
fig.update_layout(
    title='Número de viajes según tipo de lluvia (yellow)',       # MODIFICAR: Título del gráfico
    template='plotly_white',         # MODIFICAR: Tema
    legend=dict(
        orientation="h",             # MODIFICAR: Orientación de leyenda
        yanchor="bottom",
        y=-0.1,
        xanchor="center",
        x=0.5
    ),
    height=500,
    width=700,
    margin=dict(l=20, r=20, t=60, b=20)  # MODIFICAR: Márgenes
)

# Mostrar el gráfico
fig.show()


In [ ]:
fig.write_html("yellow_weather_pie.html")

In [ ]:
def agrupar_hora_zona(df, umbral = 10):
    d = df.groupby([df.tpep_pickup_datetime.dt.hour, df.PULocationID])["num_viajes"].mean().reset_index(name='num_viajes')
    d.columns = ["hora", "pick_up_id", "num_viajes"]
    d = d[d.num_viajes > umbral]
    return d

In [ ]:
df_sin_lluvia = taxi_weather[taxi_weather.rain_type == "Sin lluvia"]
df_lluvia_ligera =  taxi_weather[taxi_weather.rain_type == "Lluvia ligera"]
df_lluvia_moderada =  taxi_weather[taxi_weather.rain_type == "Lluvia moderada"]
df_lluvia_intensa = taxi_weather[taxi_weather.rain_type == "Lluvia intensa"]

In [ ]:
df_sin_lluvia = agrupar_hora_zona(df_sin_lluvia,5)
df_lluvia_ligera = agrupar_hora_zona(df_lluvia_ligera,5)
df_lluvia_moderada = agrupar_hora_zona(df_lluvia_moderada,5)
df_lluvia_intensa = agrupar_hora_zona(df_lluvia_intensa,5)

In [ ]:
import json
gdf_json = json.loads(gdf.to_json())

dataframes = [df_sin_lluvia, df_lluvia_ligera, df_lluvia_moderada,df_lluvia_intensa]
colorscales = ['Blues', 'Greens', 'Oranges', 'Reds']
names = ["Sin lluvia", "Lluvia ligera", "Lluvia moderada", "Lluvia intensa"]
num_conditions = len(dataframes)

fig = go.Figure()

for i, df in enumerate(dataframes):
    # Datos iniciales (Hora 0)
    df_hora0 = df[df['hora'] == 0]
    
    y_pos = 0.85 - (i * (0.8 / num_conditions))
    
    fig.add_trace(go.Choroplethmap(
        geojson=gdf_json,
        locations=df_hora0['pick_up_id'],
        z=df_hora0['num_viajes'],
        featureidkey="properties.id",
        colorscale=colorscales[i],
        zmin=df['num_viajes'].min(),
        zmax=df['num_viajes'].max(),
        marker_opacity=0.7,
        marker_line_width=0.5,
        name=names[i],
        showlegend=True,
        colorbar=dict(
            title=names[i],
            x=1.02,
            len=1.0 / num_conditions - 0.05,
            y=y_pos,
            yanchor="top"
        ),
    ))

frames = []

for hora in range(0, 24):
    frame_data = []
    
    # Para cada clima, actualizamos sus datos 
    for i, df in enumerate(dataframes):
        df_hora = df[df['hora'] == hora]
        
        # Cada frame debe tener una lista de diccionarios con los nuevos datos
        # type='choroplethmap' es necesario para identificar el tipo
        frame_data.append(go.Choroplethmap(
            locations=df_hora['pick_up_id'],
            z=df_hora['num_viajes']
        ))
    
    frames.append(go.Frame(
        data=frame_data,
        name=f"frame_{hora}" # Nombre del frame para referenciarlo
    ))

fig.frames = frames

steps = []
for hora in range(0, 24):
    step = dict(
        method="animate",
        args=[
            [f"frame_{hora}"], # Nombre del frame a mostrar
            dict(
                mode="immediate",
                frame=dict(duration=0, redraw=True), # Transición instantánea
                transition=dict(duration=0)
            )
        ],
        label=f"{hora:02d}:00"
    )
    steps.append(step)

sliders = [dict(
    active=0,
    yanchor="top",
    y=-0.1,
    xanchor="left",
    x=0.01,
    currentvalue=dict(
        prefix="Hora: ",
        visible=True,
        xanchor="left"
    ),
    pad={"b": 10, "t": 30},
    len=0.9,
    steps=steps
)]

In [ ]:
# === 4. LAYOUT FINAL ===
fig.update_layout(
    title="Viajes promedios realizados por Zona y tipo de lluvia (yellow)",
    map=dict(
        style="carto-positron",
        zoom=10,
        center=dict(lat=40.7128, lon=-73.9352)
    ),
    sliders=sliders,
    showlegend=True, # La leyenda ahora funciona perfectamente
    height=1200,
    margin=dict(b=100)
)
fig.write_html("yellow_weather_map.html")
fig.show()

In [ ]:
df_sin_lluvia = taxi_weather[taxi_weather.rain_type == "Sin lluvia"]
df_lluvia_ligera =  taxi_weather[taxi_weather.rain_type == "Lluvia ligera"]
df_lluvia_moderada =  taxi_weather[taxi_weather.rain_type == "Lluvia moderada"]
df_lluvia_intensa = taxi_weather[taxi_weather.rain_type == "Lluvia intensa"]

dfs = [
    (df_sin_lluvia, "Sin Lluvia", "gray"),
    (df_lluvia_ligera, "Lluvia Ligera", "skyblue"),
    (df_lluvia_moderada, "Lluvia Moderada", "dodgerblue"),
    (df_lluvia_intensa, "Lluvia Intensa", "darkred")
]

fig = go.Figure()

for df, nombre, color in dfs:

    means = df.groupby(df.tpep_pickup_datetime.dt.hour)["num_viajes"].mean()
    # 2. Añadir traza
    fig.add_trace(go.Scatter(
        x=means.index, 
        y=means.values, 
        mode='lines', 
        name=nombre, 
        line=dict(color=color, width=3) 
    ))

fig.update_layout(
    title="Número de viajes promedio por tipo de Lluvia (yellow)",
    xaxis_title="Hora del Día (0-23)",
    yaxis_title="Número de viajes promedio por hora",
    xaxis=dict(tickmode='linear', tick0=0, dtick=1), # Muestra todas las horas
    template="plotly_white",
    hovermode="x unified" 
)

fig.show()

In [ ]:
fig.write_html("yellow_weather_line.html")

In [ ]:
del taxi_weather
del taxi

# fhvhv

In [ ]:
dfs  = []
years = range(2023,2026,1)
months = [ f"{m:02d}" for m in range(1,13,1)]
for year in years:
    for month in months:
        filePath = Path(fhvhvPath, "fhvhv_tripdata_" + f"{year}-{month}.parquet") 
        if (not filePath.exists() or not filePath.is_file()):
            continue
        try:
            d = pd.read_parquet(filePath, columns = ["pickup_datetime", "PULocationID"])
        except FileNotFoundError:
            print(f"File {filePath} doesn't exist")
            continue
        dfs.append(d)
fhvhv = pd.concat(dfs)
fhvhv['pickup_datetime'] = fhvhv['pickup_datetime'].dt.floor('h')

In [ ]:
del dfs

In [ ]:
# Filtramos los unknown
fhvhv = fhvhv[ fhvhv.PULocationID < 264]
fhvhv = fhvhv[(fhvhv.pickup_datetime.dt.year >= 2022) & (fhvhv.pickup_datetime.dt.year < 2026) ]
fhvhv = fhvhv.groupby([fhvhv.pickup_datetime, fhvhv.PULocationID]).size().reset_index(name='num_viajes')

In [ ]:
fhvhv.info()

In [ ]:
fhvhv_weather = fhvhv.merge(
    weather[['datetime', 'id', 'rain']], 
    left_on=['pickup_datetime', 'PULocationID'],         
    right_on=['datetime', 'id'],            
    how='left'                                      
)
fhvhv_weather = fhvhv_weather.dropna()
fhvhv_weather = fhvhv_weather.drop("id",axis= 1)

In [ ]:
bins = [-1, 0, 2.5, 15, np.inf]
labels = ['Sin lluvia', 'Lluvia ligera', 'Lluvia moderada', 'Lluvia intensa']
fhvhv_weather['rain_type'] = pd.cut(
    fhvhv_weather['rain'],
    bins=bins,
    labels=labels
)
fhvhv_weather

In [ ]:
df = fhvhv_weather.groupby("rain_type")['num_viajes'].sum().reset_index(name='freq')
# Crear figura
fig = go.Figure(
    data=[
        go.Pie(
            labels=df['rain_type'],  # MODIFICAR: Columna para etiquetas
            values=df['freq'],     # MODIFICAR: Columna para valores
            hole=0.3,                 # MODIFICAR: Tamaño del agujero (0-1, 0 para pie completo)
            textinfo='label+percent', # MODIFICAR: Información a mostrar ('label', 'percent', 'value', 'label+percent', etc.)
            insidetextorientation='radial',  # MODIFICAR: Orientación del texto ('horizontal', 'radial', 'tangential')
            pull=[0, 0.1, 0, 0, 0],   # MODIFICAR: Separación de sectores (lista con valores 0-1)
            marker=dict(
                colors=['#636EFA', '#00CC96', '#EF553B','#AB63FA'],  # MODIFICAR: Colores personalizados
                line=dict(color='#FFFFFF', width=2)  # MODIFICAR: Borde de los sectores
            ),
            rotation=90,              # MODIFICAR: Ángulo de rotación en grados
            direction='clockwise',    # MODIFICAR: Dirección ('clockwise', 'counterclockwise')
            sort=True                 # MODIFICAR: Ordenar sectores por valor
        )
    ]
)

# MODIFICAR: Personaliza el diseño
fig.update_layout(
    title='Número de viajes según tipo de lluvia (FHVHV)',       # MODIFICAR: Título del gráfico
    template='plotly_white',         # MODIFICAR: Tema
    legend=dict(
        orientation="h",             # MODIFICAR: Orientación de leyenda
        yanchor="bottom",
        y=-0.1,
        xanchor="center",
        x=0.5
    ),
    height=500,
    width=700,
    margin=dict(l=20, r=20, t=60, b=20)  # MODIFICAR: Márgenes
)

# Mostrar el gráfico
fig.show()

In [ ]:
def agrupar_hora_zona(df, umbral = 10):
    d = df.groupby([df.pickup_datetime.dt.hour, df.PULocationID])["num_viajes"].mean().reset_index(name='num_viajes')
    d.columns = ["hora", "pick_up_id", "num_viajes"]
    d = d[d.num_viajes > umbral]
    return d

In [ ]:
df_sin_lluvia = agrupar_hora_zona(fhvhv_weather[fhvhv_weather.rain_type == "Sin lluvia"])
df_lluvia_ligera =  agrupar_hora_zona(fhvhv_weather[fhvhv_weather.rain_type == "Lluvia ligera"])
df_lluvia_moderada =  agrupar_hora_zona(fhvhv_weather[fhvhv_weather.rain_type == "Lluvia moderada"])
df_lluvia_intensa = agrupar_hora_zona(fhvhv_weather[fhvhv_weather.rain_type == "Lluvia intensa"])

In [ ]:
import json
gdf_json = json.loads(gdf.to_json())

dataframes = [df_sin_lluvia, df_lluvia_ligera, df_lluvia_moderada,df_lluvia_intensa]
colorscales = ['Blues', 'Greens', 'Oranges', 'Reds']
names = ["Sin lluvia", "Lluvia ligera", "Lluvia moderada", "Lluvia intensa"]
num_conditions = len(dataframes)

fig = go.Figure()

for i, df in enumerate(dataframes):
    # Datos iniciales (Hora 0)
    df_hora0 = df[df['hora'] == 0]
    
    y_pos = 0.85 - (i * (0.8 / num_conditions))
    
    fig.add_trace(go.Choroplethmap(
        geojson=gdf_json,
        locations=df_hora0['pick_up_id'],
        z=df_hora0['num_viajes'],
        featureidkey="properties.id",
        colorscale=colorscales[i],
        zmin=df['num_viajes'].min(),
        zmax=df['num_viajes'].max(),
        marker_opacity=0.7,
        marker_line_width=0.5,
        name=names[i],
        showlegend=True,
        colorbar=dict(
            title=names[i],
            x=1.02,
            len=1.0 / num_conditions - 0.05,
            y=y_pos,
            yanchor="top"
        ),
    ))

frames = []

for hora in range(0, 24):
    frame_data = []
    
    # Para cada clima, actualizamos sus datos 
    for i, df in enumerate(dataframes):
        df_hora = df[df['hora'] == hora]
        
        # Cada frame debe tener una lista de diccionarios con los nuevos datos
        # type='choroplethmap' es necesario para identificar el tipo
        frame_data.append(go.Choroplethmap(
            locations=df_hora['pick_up_id'],
            z=df_hora['num_viajes']
        ))
    
    frames.append(go.Frame(
        data=frame_data,
        name=f"frame_{hora}" # Nombre del frame para referenciarlo
    ))

fig.frames = frames

steps = []
for hora in range(0, 24):
    step = dict(
        method="animate",
        args=[
            [f"frame_{hora}"], # Nombre del frame a mostrar
            dict(
                mode="immediate",
                frame=dict(duration=0, redraw=True), # Transición instantánea
                transition=dict(duration=0)
            )
        ],
        label=f"{hora:02d}:00"
    )
    steps.append(step)

sliders = [dict(
    active=0,
    yanchor="top",
    y=-0.1,
    xanchor="left",
    x=0.01,
    currentvalue=dict(
        prefix="Hora: ",
        visible=True,
        xanchor="left"
    ),
    pad={"b": 10, "t": 30},
    len=0.9,
    steps=steps
)]

In [ ]:
# === 4. LAYOUT FINAL ===
fig.update_layout(
    title="Viajes promedios realizados por Zona y tipo de lluvia (FHVHV)",
    map=dict(
        style="carto-positron",
        zoom=10,
        center=dict(lat=40.7128, lon=-73.9352)
    ),
    sliders=sliders,
    showlegend=True, # La leyenda ahora funciona perfectamente
    height=1200,
    margin=dict(b=100)
)
fig.write_html("fhvhv_weather_map.html")
fig.show()

In [ ]:
df_sin_lluvia = fhvhv_weather[fhvhv_weather.rain_type == "Sin lluvia"]
df_lluvia_ligera =  fhvhv_weather[fhvhv_weather.rain_type == "Lluvia ligera"]
df_lluvia_moderada =  fhvhv_weather[fhvhv_weather.rain_type == "Lluvia moderada"]
df_lluvia_intensa = fhvhv_weather[fhvhv_weather.rain_type == "Lluvia intensa"]

dfs = [
    (df_sin_lluvia, "Sin Lluvia", "gray"),
    (df_lluvia_ligera, "Lluvia Ligera", "skyblue"),
    (df_lluvia_moderada, "Lluvia Moderada", "dodgerblue"),
    (df_lluvia_intensa, "Lluvia Intensa", "darkred")
]

fig = go.Figure()

for df, nombre, color in dfs:

    means = df.groupby(df.pickup_datetime.dt.hour)["num_viajes"].mean()
    # 2. Añadir traza
    fig.add_trace(go.Scatter(
        x=means.index, 
        y=means.values, 
        mode='lines', 
        name=nombre, 
        line=dict(color=color, width=3) 
    ))

fig.update_layout(
    title="Número de viajes promedio por tipo de Lluvia (FHVHV)",
    xaxis_title="Hora del Día (0-23)",
    yaxis_title="Número de viajes promedio por hora",
    xaxis=dict(tickmode='linear', tick0=0, dtick=1), # Muestra todas las horas
    template="plotly_white",
    hovermode="x unified" 
)

fig.show()

In [ ]:
fig.write_html("fhvhv_weather_line.html")